In [1]:
import os
import json
import time
from datasets import load_dataset
from openai import OpenAI
from tqdm import tqdm

In [14]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
secret_value_1 = user_secrets.get_secret("OPENROUTER_API_KEY")
secret_value_2 = user_secrets.get_secret("kaggle")


In [3]:
from huggingface_hub import login
login(token=secret_value_0)

In [9]:
# Setup OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=secret_value_1
)

In [5]:
MODEL_ID = "openai/gpt-oss-20b:free"

print("Loading 100 arXiv abstracts...")
dataset = load_dataset("gfissore/arxiv-abstracts-2021", split="train")
sample_papers = dataset.select(range(100))

Loading 100 arXiv abstracts...


README.md: 0.00B [00:00, ?B/s]

arxiv-abstracts.jsonl.gz:   0%|          | 0.00/940M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1999486 [00:00<?, ? examples/s]

In [10]:
synthetic_dataset = []

for paper in tqdm(sample_papers):
    # Prompt using the "Grad Student" persona for better semantic retrieval testing
    prompt = f"""
    ROLE: You are an expert PhD researcher reviewing the following paper.
    TITLE: {paper['title']}
    ABSTRACT: {paper['abstract']}

    TASK: Generate 2 specific search queries for a vector database:
    1. TECHNICAL: A question that can ONLY be answered by the specific technical methodology of this paper. (Hard retrieval test)
    2. INTENT: A natural query from a student looking for a mentor in this specific niche. (Semantic search test)

    OUTPUT: A strict JSON list of strings. No extra text.
    Example: ["Under what specific covariance uncertainty does the framework provide a closed-form solution?", "Who is applying distributionally robust optimization to receiver design?"]
    """

    try:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.4,  # Slightly higher for more diverse "student" phrasing
        )

        raw_output = response.choices[0].message.content.strip()

        # Clean potential markdown fluff
        if "```" in raw_output:
            raw_output = raw_output.split("```")[1].replace("json", "").strip()

        queries = json.loads(raw_output)

        for q in queries:
            synthetic_dataset.append(
                {
                    "query": q,
                    "correct_paper_id": paper["id"],
                    "original_title": paper["title"],
                }
            )

    except Exception as e:
        print(f"\nError on paper {paper['id']}: {e}")

    # OpenRouter Free Tier (April 2026) has a 20 request/min limit
    time.sleep(3.1)

  1%|          | 1/100 [00:07<12:18,  7.46s/it]


Error on paper 0704.0002: Invalid \escape: line 1 column 56 (char 55)


  4%|▍         | 4/100 [00:30<12:11,  7.62s/it]


Error on paper 0704.0005: Invalid \escape: line 1 column 69 (char 68)


 10%|█         | 10/100 [01:08<09:09,  6.11s/it]

...

 78%|███████▊  | 78/100 [07:54<01:59,  5.45s/it]


Error on paper 0704.0079: Invalid \escape: line 1 column 155 (char 154)


 86%|████████▌ | 86/100 [08:42<01:18,  5.57s/it]


Error on paper 0704.0087: Invalid \escape: line 1 column 45 (char 44)


100%|██████████| 100/100 [10:05<00:00,  6.05s/it]


In [20]:
# Save for the Evaluation script
os.makedirs("/kaggle/working/data", exist_ok=True)
with open("/kaggle/working/data/synthetic_eval_set.json", "w") as f:
    json.dump(synthetic_dataset, f, indent=4)

print(f"\n✅ Created {len(synthetic_dataset)} queries. Ready for embedding evaluation!")


✅ Created 179 queries. Ready for embedding evaluation!


In [15]:
import kagglehub

kagglehub.login()

In [17]:
import kagglehub

handle = 'AmirBnsl/synthetic_arxiv_abstract_dataset'
local_dataset_dir = '../data/synthetic_eval_set.json'

# Create a new dataset
kagglehub.dataset_upload(handle, local_dataset_dir)



Uploading Dataset https://api.kaggle.com/datasets/AmirBnsl/synthetic_arxiv_abstract_dataset ...
Starting upload for file ../data/synthetic_eval_set.json


Uploading: 100%|██████████| 62.4k/62.4k [00:00<00:00, 288kB/s]

Upload successful: ../data/synthetic_eval_set.json (61KB)


HTTPError: 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/CreateDatasetVersion